# Task 2 - detection

Needs Task 1 done (`configs/data.yaml` present). **Run All**, or run B0 first and
the training cell when you have time for it.

| Run | Model | What it is for |
| --- | --- | --- |
| B0 | COCO weights, no fine-tune | the floor. Finds only stop signs, so cone and barrier AP are 0 by construction |
| B1 | `yolo26n` fine-tuned, 640 | main deployable candidate |
| B1b | `yolo11n` fine-tuned, 640 | other family, for the latency pick in Task 4 |
| B2 | `yolo26s` fine-tuned, 960 | accuracy ceiling |

Every variant is scored by the same evaluator, including B0 - `model.val()` cannot
score an 80-class model against a 3-class set, and using two different metrics
would make the comparison meaningless.

In [ ]:
# ---- edit this ----
RUNS = ["B1"]        # add "B1b", "B2" when you want them
EPOCHS = 100         # patience=20 usually stops it earlier
WORKERS = 6          # drop to 2 if you hit WinError 1455 (paging file too small)
DEVICE = ""          # "" = auto, or "cpu", or "0"
# -------------------

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from IPython.display import Image, display

from common.config import load_yaml
from common.notebook import sh   # runs a step and streams its output live

data_yaml = REPO / "configs/data.yaml"
assert data_yaml.exists(), "run the Task 1 notebook first"
DEVICE_ARGS = ["--device", DEVICE] if DEVICE else []
RUN_SPECS = load_yaml("runs.yaml")

print(data_yaml.read_text())

In [ ]:
# The floor. Everything below has to beat the stop sign number.
sh("-m", "eval.det_eval", "--run", "B0", *DEVICE_ARGS)

In [ ]:
# Train and score each run. One row of output per epoch, as it happens.
# exist_ok=True keeps the weights at runs/<name>/ so a re-run replaces them
# instead of creating runs/B12, runs/B13 and so on.
for run in RUNS:
    spec = {**RUN_SPECS["defaults"], **RUN_SPECS["runs"][run]}
    print()
    print(f"=== {run}: {spec.get('purpose', '')} ===", flush=True)
    sh(
        "-m", "detection.train", "--run", run,
        "--set", f"epochs={EPOCHS}",
        "--set", f"workers={WORKERS}",
        "--set", "exist_ok=True",
        *DEVICE_ARGS,
    )
    sh(
        "-m", "eval.det_eval",
        "--weights", f"runs/{run}/weights/best.pt",
        "--name", run,
        "--imgsz", str(spec["imgsz"]),
        "--cross-check",
        *DEVICE_ARGS,
    )

In [ ]:
# Per-epoch history. Ultralytics appends a row after every epoch, so this also
# works as a progress monitor: if you started training in a terminal instead,
# re-run this cell whenever you want to see where it is.
import pandas as pd

hist = pd.read_csv(f"runs/{RUNS[0]}/results.csv")
hist.columns = [c.strip() for c in hist.columns]
cols = [c for c in hist.columns if "mAP50" in c or "loss" in c]
print(f"{len(hist)} epochs done")
display(hist[["epoch", *cols]].tail(12))
display(Image(f"runs/{RUNS[0]}/results.png", width=1000))

In [ ]:
# Everything scored so far.
import pandas as pd

df = pd.read_csv("results/detection.csv")
display(df)

In [ ]:
# Per-class detail, including recall by ground-truth depth - this is where far
# cones either show up or do not, and it drives the input-size decision.
import json

for run in RUNS:
    path = REPO / f"results/det_eval_{run}.json"
    if not path.exists():
        continue
    detail = json.loads(path.read_text())
    print(f"\n{run}   mAP50 {detail['mAP50']:.3f}   mAP50-95 {detail['mAP50_95']:.3f}")
    for name, m in detail["per_class"].items():
        print(f"  {name:<10} AP50 {m['AP50']:.3f}  AP50-95 {m['AP50_95']:.3f}  gt {m['gt_instances']}")
        for rng, v in m["recall_by_range"].items():
            if v["recall"] is not None:
                print(f"{'':<14}{rng:<10} recall {v['recall']:.2f}  (n={v['gt']})")

In [ ]:
# Diagnose a class that scores zero. Three checks, cheapest first.
import json

import cv2

from common.config import class_names
from common.env import dataset_root_for_cli
from common.labels import load_split

run = RUNS[0]
detail = json.loads((REPO / f"results/det_eval_{run}.json").read_text())

# 1. Did the model predict this class at all, or predict it badly?
for name, m in detail["per_class"].items():
    print(f"{name:<10} gt {m['gt_instances']:>4}   predictions {m['predictions']:>6}   AP50 {m['AP50']:.3f}")

# 2. Does ultralytics' own validator agree? If it disagrees, the evaluator is
#    at fault; if it agrees, the model or the labels are.
print()
print("ultralytics val:", detail.get("ultralytics_val", "not run - add --cross-check"))

# 3. Look at it. Grey = ground truth, colour = prediction at a low threshold.
CLASS_TO_CHECK = 1   # 0 cone, 1 barrier, 2 stop_sign
names = class_names()
root = dataset_root_for_cli()
records = [r for r in load_split(root, "val") if CLASS_TO_CHECK in r["classes"].tolist()]
print(f"{len(records)} val frames contain {names[CLASS_TO_CHECK]}")

from ultralytics import YOLO

from common.draw import draw_detections

model = YOLO(f"runs/{run}/weights/best.pt")
dest = REPO / "results/diagnose"
dest.mkdir(parents=True, exist_ok=True)
for rec in records[:3]:
    img = cv2.imread(str(rec["image"]))
    for (x1, y1, x2, y2), cls in zip(rec["boxes"], rec["classes"]):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (190, 190, 190), 2)
        cv2.putText(img, f"gt {names[cls]}", (int(x1), int(y2) + 14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (190, 190, 190), 1, cv2.LINE_AA)
    res = model.predict(str(rec["image"]), conf=0.05, verbose=False)[0]
    if len(res.boxes):
        draw_detections(img, res.boxes.xyxy.cpu().numpy(),
                        res.boxes.cls.cpu().numpy().astype(int), names,
                        confs=res.boxes.conf.cpu().numpy())
    out = dest / f"{rec['image'].stem}.jpg"
    cv2.imwrite(str(out), img)
    print(f"{rec['image'].name}: {len(rec['boxes'])} gt, {len(res.boxes)} pred at conf 0.05")
    display(Image(str(out), width=950))

In [ ]:
# What it actually draws on held-out frames.
from common.env import dataset_root_for_cli

run = RUNS[0]
sh(
    "-m", "detection.infer",
    "--weights", f"runs/{run}/weights/best.pt",
    "--source", str(dataset_root_for_cli() / "images" / "val"),
    "--limit", "6",
    "--dest", f"results/predictions/{run}",
    *DEVICE_ARGS,
)

for p in sorted((REPO / f"results/predictions/{run}").glob("*.jpg"))[:6]:
    display(Image(str(p), width=900))

## Done when

- every fine-tuned run beats B0 on **all three** classes (B0 was stop sign AP50 0.745);
- `--cross-check` agrees with the custom evaluator within a point or two - small gaps are
  expected, ultralytics letterboxes and rectangles differently;
- stop sign AP has not collapsed. If it has, suspect the pseudo-labels from Task 1.

`runs/B1/weights/best.pt` is what Tasks 3 and 4 start from. Weights are git-ignored, so
copy that file by hand if the next task runs on the other machine.